In [3]:
import os
import time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# -------------------------------
# DATASET DEFINITION
# -------------------------------
class UCFCrimeDataset(Dataset):
    def __init__(self, base_dir):
        self.files = []
        self.labels = []
        # Fallback dictionary if directory is empty or path doesn't exist yet
        if os.path.exists(base_dir) and os.path.isdir(base_dir):
            categories = sorted(os.listdir(base_dir))
            self.label_map = {cat: i for i, cat in enumerate(categories)}
            
            for cat in categories:
                cat_path = os.path.join(base_dir, cat)
                if not os.path.isdir(cat_path):
                    continue
                for f in os.listdir(cat_path):
                    if f.endswith('.npy'):
                        self.files.append(os.path.join(cat_path, f))
                        self.labels.append(self.label_map[cat])
        else:
            # Standard 14-class mapping structure for UCF-Crime for evaluation fallback
            classes = ["Abuse", "Arrest", "Arson", "Assault", "Burglary", "Explosion", 
                       "Fighting", "Normal", "RoadAccidents", "Robbery", "Shooting", 
                       "Shoplifting", "Stealing", "Vandalism"]
            self.label_map = {cat: i for i, cat in enumerate(classes)}
            print(f"Warning: Base directory '{base_dir}' not found. Using standard 14-class template mapping.")
            
        print("Loaded Classes Mapping:", self.label_map)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        x = np.load(self.files[idx])  
        y = self.labels[idx]
        return torch.from_numpy(x).float(), torch.tensor(y, dtype=torch.long)

# -------------------------------
# MODEL ARCHITECTURE
# -------------------------------
class Attention(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.attn = nn.Linear(input_dim, 1)

    def forward(self, x):
        weights = torch.softmax(self.attn(x), dim=1)  
        weighted = x * weights
        return torch.sum(weighted, dim=1) 

class CrimeBiLSTM_Attention(nn.Module):
    def __init__(self, input_dim=512, hidden_dim=256, num_classes=14):
        super().__init__()
        self.lstm = nn.LSTM(
            input_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True,
            num_layers=2,           
            dropout=0.3             
        )
        self.attention = Attention(hidden_dim * 2)
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)  
        context = self.attention(lstm_out) 
        return self.classifier(context)

# -------------------------------
# EVALUATION METRICS PIPELINE
# -------------------------------
def evaluate_dataset_performance(model, data_loader, device, inverse_label_map):
    print("\n==================================================")
    print("      RUNNING BATCH DATASET EVALUATION            ")
    print("==================================================")
    
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for x, y in data_loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            _, predicted = torch.max(outputs, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
            
    if len(all_labels) == 0:
        print("Evaluation aborted: No evaluation data files found in directory path.")
        return

    # Compute Parameters
    accuracy = accuracy_score(all_labels, all_preds)
    print(f"\nOverall Test Accuracy: {accuracy * 100:.2f}%")
    
    print("\nDetailed Classification Report:")
    target_names = [inverse_label_map[i] for i in range(len(inverse_label_map))]
    print(classification_report(all_labels, all_preds, target_names=target_names, zero_division=0))
    
    print("\nRaw Confusion Matrix Layout:")
    cm = confusion_matrix(all_labels, all_preds)
    print(cm)

# -------------------------------
# REAL WORLD INFERENCE SIMULATION
# -------------------------------
def run_real_world_inference_simulation(model, device, inverse_label_map):
    print("\n==================================================")
    print("      REAL WORLD CCTV STREAMING SIMULATION       ")
    print("==================================================")
    
    model.eval()
    
    # Simulate an incoming feature stream from a live camera
    # Shape: [Batch size = 1, Time Sequences/Segments = 32, Feature Size = 512]
    simulated_segments = 32
    mock_live_stream_frame_features = torch.randn(1, simulated_segments, 512).to(device)
    
    print(f"Receiving live camera stream buffer segment tensor of size: {list(mock_live_stream_frame_features.shape)}")
    
    # Track execution latency parameters for real-time compliance
    start_time = time.perf_counter()
    with torch.no_grad():
        raw_outputs = model(mock_live_stream_frame_features)
        probabilities = torch.softmax(raw_outputs, dim=1)
        confidence_score, predicted_idx = torch.max(probabilities, 1)
    end_time = time.perf_counter()
    
    latency_ms = (end_time - start_time) * 1000
    predicted_class = inverse_label_map[predicted_idx.item()]
    confidence = confidence_score.item() * 100
    
    print(f"Inference Latency: {latency_ms:.2f} ms")
    print(f"Frames Per Second Processing Capability: {1000 / latency_ms:.2f} FPS")
    
    # Security automation logic: Thresholding deployment system
    ALERT_THRESHOLD = 75.0  # Require 75% system confidence to trigger system alerts
    
    print(f"\n[ criDe Engine Diagnosis ] Predicted Activity: '{predicted_class}' with {confidence:.2f}% confidence.")
    
    if predicted_class != "Normal" and confidence >= ALERT_THRESHOLD:
        print(f"🚨 CRITICAL ALERT: Suspicious activity [{predicted_class}] detected! Triggering alert notification system.")
    elif predicted_class != "Normal" and confidence < ALERT_THRESHOLD:
        print(f"⚠️ LOGGED INCIDENT: Low confidence threat classification ({confidence:.2f}%). Recording to database for human audit.")
    else:
        print("✅ SYSTEM STABLE: Monitoring live sequence feed... Activity normal.")

# -------------------------------
# MAIN EXECUTION SCRIPT
# -------------------------------
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    DATA_PATH = "../extracted_features"
    MODEL_WEIGHTS_PATH = "best_bilstm_model.pth"
    
    # Initialize dataset object to gather target mapping configuration
    dataset = UCFCrimeDataset(DATA_PATH)
    inverse_label_map = {v: k for k, v in dataset.label_map.items()}
    num_classes = len(dataset.label_map)
    
    # Instantiate Model
    model = CrimeBiLSTM_Attention(num_classes=num_classes).to(device)
    
    # Safely load the trained weights if they exist
    if os.path.exists(MODEL_WEIGHTS_PATH):
        model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH, map_location=device))
        print(f"Successfully loaded trained weights file from: {MODEL_WEIGHTS_PATH}")
    else:
        print(f"Weights file '{MODEL_WEIGHTS_PATH}' not found. Performing test run with random initialization parameters.")

    # Create Evaluation Loader (num_workers=0 ensures strict safety on local Windows configurations)
    eval_loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=True)
    
    # Test Parameter 1: Complete Evaluation Metrics Loop
    if len(dataset) > 0:
        evaluate_dataset_performance(model, eval_loader, device, inverse_label_map)
    
    # Test Parameter 2: Execution Latency & Real-World Decision Pipeline Test
    run_real_world_inference_simulation(model, device, inverse_label_map)

if __name__ == "__main__":
    main()

Loaded Classes Mapping: {'Abuse': 0, 'Arrest': 1, 'Arson': 2, 'Assault': 3, 'Burglary': 4, 'Explosion': 5, 'Fighting': 6, 'NormalVideos': 7, 'RoadAccidents': 8, 'Robbery': 9, 'Shooting': 10, 'Shoplifting': 11, 'Stealing': 12, 'Vandalism': 13}
Successfully loaded trained weights file from: best_bilstm_model.pth

      RUNNING BATCH DATASET EVALUATION            

Overall Test Accuracy: 59.19%

Detailed Classification Report:
               precision    recall  f1-score   support

        Abuse       0.50      0.08      0.14        48
       Arrest       0.00      0.00      0.00        45
        Arson       0.67      0.10      0.17        41
      Assault       0.00      0.00      0.00        47
     Burglary       0.45      0.10      0.17        87
    Explosion       0.00      0.00      0.00        29
     Fighting       0.20      0.07      0.10        45
 NormalVideos       0.62      0.97      0.76       800
RoadAccidents       0.46      0.68      0.55       127
      Robbery       0

In [4]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from torchvision import models, transforms
from collections import deque

# -------------------------------
# MODEL ARCHITECTURES
# -------------------------------
class Attention(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.attn = nn.Linear(input_dim, 1)

    def forward(self, x):
        # x shape: [Batch, Seq_Len, Dim]
        weights = torch.softmax(self.attn(x), dim=1)  
        weighted = x * weights
        return torch.sum(weighted, dim=1) 

class CrimeBiLSTM_Attention(nn.Module):
    def __init__(self, input_dim=512, hidden_dim=256, num_classes=14):
        super().__init__()
        self.lstm = nn.LSTM(
            input_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True,
            num_layers=2,           
            dropout=0.3             
        )
        self.attention = Attention(hidden_dim * 2)
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)  
        context = self.attention(lstm_out) 
        return self.classifier(context)

# -------------------------------
# LIVE INFERENCE ENGINE
# -------------------------------
def run_live_surveillance():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Initializing criDe live engine on device: {device}")

    # 1. Define Class Labels (Ensure this matches your training label map ordering)
    classes = ["Abuse", "Arrest", "Arson", "Assault", "Burglary", "Explosion", 
               "Fighting", "Normal", "RoadAccidents", "Robbery", "Shooting", 
               "Shoplifting", "Stealing", "Vandalism"]
    
    # 2. Initialize and Load the BiLSTM Model
    num_classes = len(classes)
    model = CrimeBiLSTM_Attention(num_classes=num_classes).to(device)
    
    MODEL_PATH = "best_bilstm_model.pth"
    if os.path.exists(MODEL_PATH):
        model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
        print(f"Loaded trained weights successfully from {MODEL_PATH}")
    else:
        print(f"Warning: '{MODEL_PATH}' not found. Running inference engine with random initialization parameters.")
    model.eval()

    # 3. Initialize Pre-trained ResNet18 for Real-Time Spatial Feature Extraction
    # ResNet18's average pooling layer naturally outputs a 512-dimensional vector
    resnet = models.resnet18(pretrained=True)
    resnet.fc = nn.Identity() # Remove final classification layer to extract raw 512 features
    resnet = resnet.to(device)
    resnet.eval()

    # 4. Image Preprocessing Transform for the Feature Extractor
    preprocess = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    # 5. Setup Sliding Window Buffer
    SEQUENCE_LENGTH = 32  # Number of temporal segments your model expects
    feature_buffer = deque(maxlen=SEQUENCE_LENGTH)

    # 6. Initialize Video Capture (0 represents the primary system webcam)
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Error: Could not open video stream or webcam hardware resource.")
        return

    print("Live surveillance window active. Press 'q' to terminate stream safely.")

    current_prediction = "Initializing Buffer..."
    confidence_text = ""

    while True:
        ret, frame = cap.read()
        if not ret:
            print("Failed to grab camera frame interface link.")
            break

        # Clone raw frame for processing, keeping original intact for visual output
        input_frame = frame.copy()
        
        # Extract features for the current frame
        tensor_frame = preprocess(input_frame).unsqueeze(0).to(device)
        with torch.no_grad():
            # Output shape: [1, 512]
            frame_features = resnet(tensor_frame).squeeze(0).cpu().numpy()
        
        # Append the new 512-dimensional vector to our rolling timeline queue
        feature_buffer.append(frame_features)

        # Once the temporal buffer is completely full, run sequence prediction
        if len(feature_buffer) == SEQUENCE_LENGTH:
            # Convert sliding window queue into a structured sequence tensor
            sequence_tensor = torch.tensor(np.array(feature_buffer), dtype=torch.float).unsqueeze(0).to(device)
            
            with torch.no_grad():
                # Pass sequence [Batch=1, Seq=32, Features=512] through the BiLSTM
                raw_logits = model(sequence_tensor)
                probabilities = torch.softmax(raw_logits, dim=1)
                confidence, predicted_idx = torch.max(probabilities, 1)
                
            current_prediction = classes[predicted_idx.item()]
            confidence_text = f"{confidence.item() * 100:.1f}%"

        # 7. Render UI Text Overlays on the Live Window
        # Set background status border colors based on safety profile
        text_color = (0, 255, 0) if current_prediction == "Normal" or "Initializing" in current_prediction else (0, 0, 255)
        
        display_string = f"Status: {current_prediction} {confidence_text}"
        
        cv2.putText(
            frame, 
            display_string, 
            (20, 40), 
            cv2.FONT_HERSHEY_SIMPLEX, 
            0.8, 
            text_color, 
            2, 
            cv2.LINE_AA
        )
        
        # Render frame composition output window
        cv2.imshow("criDe - Live Video Threat Analytics Engine", frame)

        # Break the loop immediately if the 'q' key is pressed
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # Resource Cleanup
    cap.release()
    cv2.destroyAllWindows()
    print("Camera stream closed down safely.")

if __name__ == "__main__":
    run_live_surveillance()

Initializing criDe live engine on device: cuda
Loaded trained weights successfully from best_bilstm_model.pth
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\navee/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


C:\Users\navee\AppData\Roaming\Python\Python314\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\navee\AppData\Roaming\Python\Python314\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 44.7M/44.7M [00:10<00:00, 4.33MB/s]


Live surveillance window active. Press 'q' to terminate stream safely.
Camera stream closed down safely.
